In [ ]:
# Tagging at threshold, analytic:  the phone network and the tagger network against background
# Devices:  N phones and N taggers, an area each (area x efficiency, efficiency taken as 1), from model.yaml.
# Naive signal model:  the year's detectable showers at Phi_D, each at exactly density Phi_D.
# Each device fires with 1 - exp(-lambda), so the number of fired devices out of N is binomial.

import numpy as np
import matplotlib.pyplot as plt
from math import comb
from model import alpha, gamma, epsilon, beta, eta, lmbda, C, phi_D, YEAR
from model import mu_flux, n_phones, A_phone, dt_phone, n_taggers, A_tagger, dt_tagger

phid = phi_D[0]                    # muons / m^2
# override phid here as needed:
phid = 50;

# the detectable rate, closed form (full_analytic)
E_min = epsilon * (2 * np.pi * lmbda**2 * phid / beta)**(1 / eta)
R = 2 * C * np.pi**2 * alpha * (lmbda * eta)**2 * (E_min / epsilon)**(1 - gamma) / (gamma - 1)**3
n_events = R * YEAR                # showers per year at or above phid

print(f"Phi_D {phid:g} / m^2:  E_min {E_min:.3g} eV,  {n_events:.2f} showers / yr")

In [ ]:
# override model.yaml here as needed:
n_phones = 1000
dt_phone = 0.1

# Phones
lam_b = mu_flux * A_phone * dt_phone     # background hits per phone per window
lam_s = phid * A_phone                   # shower hits per phone
p_b = 1 - np.exp(-lam_b)                 # a phone fires, background only
p_s = 1 - np.exp(-(lam_s + lam_b))       # a phone fires, shower plus background

# fired phones k = 0..K in one window (K well above anything likely)
K = 30
pmf_s = [comb(n_phones, k) * p_s**k * (1 - p_s)**(n_phones - k) for k in range(K + 1)]
pmf_b = [comb(n_phones, k) * p_b**k * (1 - p_b)**(n_phones - k) for k in range(K + 1)]

print(f"{n_phones} phones of {A_phone:g} m^2,  per phone:  lambda_s {lam_s:.2e},  lambda_b {lam_b:.2e}")
print(f"expected fired phones per window:  shower {n_phones * p_s:.2f},  background {n_phones * p_b:.2f}")

k = np.arange(K + 1)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.step(k, pmf_s, where="mid", label="shower at $\\Phi_D$ (+ background)")
ax.step(k, pmf_b, where="mid", label="background only")
ax.set_yscale("log"); ax.set_ylim(1e-12, 1.5); ax.set_xlim(-0.5, 5.5)
ax.set_xlabel(f"fired phones (of {n_phones}) in {dt_phone:g} s"); ax.set_ylabel("probability")
ax.set_title(f"Phone network, $\\Phi_D$ = {phid:g}")
ax.legend(); plt.tight_layout(); plt.savefig(f"pdf/tagging_phones_{phid:g}.pdf"); plt.show()

In [ ]:
# Taggers

#override model.yaml as needed
n_taggers=10

lam_b = mu_flux * A_tagger * dt_tagger   # background hits per tagger per window
lam_s = phid * A_tagger                  # shower hits per tagger
q_b = 1 - np.exp(-lam_b)
q_s = 1 - np.exp(-(lam_s + lam_b))

pmf_ts = [comb(n_taggers, k) * q_s**k * (1 - q_s)**(n_taggers - k) for k in range(n_taggers + 1)]
pmf_tb = [comb(n_taggers, k) * q_b**k * (1 - q_b)**(n_taggers - k) for k in range(n_taggers + 1)]

print(f"{n_taggers} taggers of {A_tagger:g} m^2 ({n_taggers * A_tagger:.3g} m^2 in all),  "
      f"single rate {mu_flux * A_tagger:.2f} Hz,  lambda_b {lam_b:.2e} per window")
print(f"a tagger fires at Phi_D with probability {q_s:.3f}")

k = np.arange(n_taggers + 1)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.step(k, pmf_ts, where="mid", label="shower at $\\Phi_D$ (+ background)")
ax.step(k, pmf_tb, where="mid", label="background only")
ax.set_yscale("log"); ax.set_ylim(1e-40, 1.5)
ax.set_xlabel(f"fired taggers (of {n_taggers}) in {dt_tagger * 1e6:g} $\\mu$s"); ax.set_ylabel("probability")
ax.set_title(f"Tagger network, $\\Phi_D$ = {phid:g}")
ax.legend(); plt.tight_layout(); plt.savefig(f"pdf/tagging_taggers_{phid:g}.pdf"); plt.show()

In [ ]:
# (1) Phones alone (untagged) inclusive counts, assumes fixed (not sliding) timing windows for phone DAQ.

print(f"Phones alone, Phi_D {phid:g}:  {n_phones} phones, {dt_phone:g} s")
print(f"{'m':>3s}  {'P(k>=m) shower':>14s}  {'shower / yr':>11s}  {'background / yr':>15s}")
m_phone, sig_phone, bkg_phone = [], [], []
for m in range(1, 9):
    P_s = sum(pmf_s[m:])
    shower = n_events * P_s
    background = YEAR / dt_phone * sum(pmf_b[m:])
    m_phone.append(m); sig_phone.append(shower); bkg_phone.append(background)
    print(f"{m:3d}  {P_s:14.3g}  {shower:11.3g}  {background:15.3g}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(m_phone, sig_phone, "o-", label="shower (+ background in its window)")
ax.plot(m_phone, bkg_phone, "s--", label="background")
ax.set_yscale("log")
ax.set_xlabel("fired phones required, m"); ax.set_ylabel("per year")
ax.set_title(f"Phones alone ({n_phones}, {dt_phone:g} s), $\\Phi_D$ = {phid:g}")
ax.legend(); plt.tight_layout(); plt.savefig(f"pdf/tagging_phones_per_year_{phid:g}.pdf"); plt.show()

In [ ]:
# (2) Taggers:  counts per year requiring at least m fired taggers;
# combinatoric background with the sliding window

m_op = 3                                 # operating point

# for sliding window calculation:
pmf_tb1 = [comb(n_taggers - 1, k) * q_b**k * (1 - q_b)**(n_taggers - 1 - k) for k in range(n_taggers)]
print(f"Taggers, Phi_D {phid:g}:  {n_taggers} taggers, {dt_tagger * 1e6:g} us")
print(f"{'m':>3s}  {'P(k>=m) shower':>14s}  {'shower / yr':>11s}  {'combinatoric / yr':>17s}")
m_tag, sig_tag, bkg_tag = [], [], []
for m in range(1, n_taggers + 1):
    P_s = sum(pmf_ts[m:])
    shower = n_events * P_s
    # fixed timing window (simple but not fully accurate):
    #background = YEAR / dt_tagger * sum(pmf_tb[m:])
    # sliding window: singles rate times rate of m-1 within the window.
    background = YEAR * n_taggers * mu_flux * A_tagger * sum(pmf_tb1[m - 1:])
    m_tag.append(m); sig_tag.append(shower); bkg_tag.append(background)
    print(f"{m:3d}  {P_s:14.3g}  {shower:11.3g}  {background:17.3g}")

tags_shower = sig_tag[m_op - 1]          # shower tags per year at the operating point
tags_combinatoric = bkg_tag[m_op - 1]    # combinatoric tags per year
print(f"\noperating point m = {m_op}:  {tags_shower:.3g} shower tags / yr,  {tags_combinatoric:.3g} combinatoric / yr")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(m_tag, sig_tag, "o-", label="shower")
ax.plot(m_tag, bkg_tag, "s--", label="combinatoric")
ax.axvline(m_op, color="0.5", linewidth=0.8)
ax.annotate(f"m = {m_op}:  {tags_shower:.3g} shower,  {tags_combinatoric:.2g} combinatoric / yr",
            xy=(m_op, tags_shower), xytext=(15, 25), textcoords="offset points")
ax.set_yscale("log"); ax.set_ylim(1e-6, None)
ax.set_xlabel("fired taggers required, m"); ax.set_ylabel("per year")
ax.set_title(f"Taggers ({n_taggers} of {A_tagger:g} m$^2$, {dt_tagger * 1e6:g} $\\mu$s), $\\Phi_D$ = {phid:g}")
ax.legend(); plt.tight_layout(); plt.savefig(f"pdf/tagging_taggers_per_year_{phid:g}.pdf"); plt.show()

In [ ]:
# (3) Phones read out at each tag

print(f"Phones at the tags, Phi_D {phid:g}:  {tags_shower:.3g} shower tags / yr (m = {m_op}),  "
      f"{n_phones} phones, {dt_phone:g} s")
print(f"{'j':>3s}  {'shower / yr':>11s}  {'background / yr':>15s}")
j_phone, sig_tagged, bkg_tagged = [], [], []
for j in range(1, 9):
    shower = tags_shower * sum(pmf_s[j:]) + tags_combinatoric * sum(pmf_b[j:])
    background = tags_shower * sum(pmf_b[j:]) + tags_combinatoric * sum(pmf_b[j:])
    j_phone.append(j); sig_tagged.append(shower); bkg_tagged.append(background)
    print(f"{j:3d}  {shower:11.3g}  {background:15.3g}")
print(f"\nfired phones per year, summed over all tags:  shower {(tags_shower * p_s + tags_combinatoric * p_b) * n_phones:.3g},  "
      f"background {(tags_shower + tags_combinatoric) * n_phones * p_b:.3g}")
print(f"combinatoric tags:  {tags_combinatoric:.2g} / yr, with {tags_combinatoric * n_phones * p_b:.2g} fired phones / yr")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(j_phone, sig_tagged, "o-", label="shower (all tags)")
ax.plot(j_phone, bkg_tagged, "s--", label="background")
ax.set_yscale("log"); ax.set_ylim(1e-6, None)
ax.set_xlabel("fired phones required at a tag, j"); ax.set_ylabel("per year")
ax.set_title(f"Phones at the tags ({n_phones}, {dt_phone:g} s;  taggers m = {m_op}), $\\Phi_D$ = {phid:g}")
ax.legend(); plt.tight_layout(); plt.savefig(f"pdf/tagging_phones_at_tags_{phid:g}.pdf"); plt.show()

In [ ]:
# Significance:  the fired phones summed over the year's tags (shower and combinatoric), against the background
# expected in those windows

from math import exp, log, lgamma
from statistics import NormalDist

b = (tags_shower + tags_combinatoric) * n_phones * p_b
n_exp = (tags_shower * p_s + tags_combinatoric * p_b) * n_phones
s = n_exp - b
n_obs = round(n_exp)
p_value = sum(exp(k * log(b) - b - lgamma(k + 1)) for k in range(n_obs, n_obs + 200))
sigma = NormalDist().inv_cdf(1 - p_value)

print(f"{tags_shower:.3g} shower tags / yr (m = {m_op}),  {n_phones} phones,  {dt_phone:g} s")
print(f"fired phones per year:  signal {s:.3g},  background {b:.3g},  observed (typical) {n_obs}")
print(f"P(n >= {n_obs} | b) = {p_value:.2g}:  {sigma:.1f} sigma")

In [ ]:
# Summary at the operating point as a LaTeX table:  analysis/tex/tagging_summary_<phid>.tex, input by docs/tagger.tex
rows = [
    ("showers per year, every one at $\\Phi_D$", f"{n_events:.1f}"),
    (f"taggers ({n_taggers} of {A_tagger:g} m$^2$, {dt_tagger * 1e6:g} $\\mu$s), at least {m_op} fired", ""),
    ("\\quad shower tags per year", f"{tags_shower:.1f}"),
    ("\\quad combinatoric tags per year", f"{tags_combinatoric:.2g}"),
    (f"phones at the tags ({n_phones} of {A_phone * 1e4:g} cm$^2$, {dt_phone:g} s), fired per year", ""),
    ("\\quad signal", f"{s:.1f}"),
    ("\\quad background", f"{b:.2f}"),
    ("significance", f"{sigma:.1f} $\\sigma$"),
]

lines = [r"\begin{tabular}{lr}", r"\hline", f"$\\Phi_D = {phid:g}$ m$^{{-2}}$, one year & \\\\", r"\hline"]
for what, value in rows:
    lines.append(f"{what} & {value} \\\\")
lines += [r"\hline", r"\end{tabular}"]
open(f"tex/tagging_summary_{phid:g}.tex", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))